# 01 — Sentinel-2 数据下载
**数据源：Microsoft Planetary Computer**

本笔记本从 Planetary Computer 下载 NSW 测试区的 Sentinel-2 L2A 影像，
计算 NDVI、NDWI、BSI、SAVI 四个指数，并保存为 GeoTIFF。

测试区：悉尼西部农业区（147°E–149°E，34°S–32.5°S）
时间段：2022年1月

## 1. 导入包

In [ ]:
import sys
import os
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from pystac_client import Client
import stackstac
import rasterio
from rasterio.transform import from_bounds

from src.data_utils import load_config, ensure_dir, print_array_info
from src.indices import calculate_all_indices

print('所有包导入成功')

## 2. 加载配置

In [ ]:
cfg = load_config('../configs/config.yaml')

# 使用测试小区域
bbox = cfg['region']['bbox_test']
BBOX = [bbox['lon_min'], bbox['lat_min'], bbox['lon_max'], bbox['lat_max']]

TIME_RANGE = '2022-01-01/2022-01-31'
CLOUD_MAX  = cfg['sentinel2']['cloud_cover_max']
STAC_URL   = cfg['sentinel2']['stac_endpoint']

print(f'研究区 BBOX : {BBOX}')
print(f'时间范围    : {TIME_RANGE}')
print(f'最大云量    : {CLOUD_MAX}%')
print(f'STAC 端点   : {STAC_URL}')

## 3. 搜索 Sentinel-2 场景

In [ ]:
catalog = Client.open(STAC_URL)
print(f'已连接: {catalog.title}')

search = catalog.search(
    collections=['sentinel-2-l2a'],
    bbox=BBOX,
    datetime=TIME_RANGE,
    query={'eo:cloud_cover': {'lt': CLOUD_MAX}}
)

items = list(search.items())
print(f'\n找到场景数: {len(items)}')
print(f'{"场景ID":<60} {"日期":<12} {"云量":>6}')
print('-' * 82)
for item in items[:10]:
    date  = item.datetime.strftime('%Y-%m-%d')
    cloud = item.properties.get('eo:cloud_cover', 'N/A')
    print(f'{item.id:<60} {date:<12} {cloud:>6.1f}%')

## 4. 选取云量最低的场景

In [ ]:
# 按云量排序，选最低的一景
items_sorted = sorted(
    items,
    key=lambda x: x.properties.get('eo:cloud_cover', 999)
)

best = items_sorted[0]
print(f'选取场景  : {best.id}')
print(f'日期      : {best.datetime.strftime("%Y-%m-%d")}')
print(f'云量      : {best.properties["eo:cloud_cover"]:.2f}%')
print(f'可用波段  : {list(best.assets.keys())[:15]}')

## 5. 加载波段数据

In [ ]:
BANDS = ['B02', 'B03', 'B04', 'B08', 'B8A', 'B11']

stack = stackstac.stack(
    [best],
    assets=BANDS,
    bounds_latlon=BBOX,
    resolution=10,
    dtype='float32'
)

print(f'数据维度: {stack.dims}')
print(f'数据形状: {stack.shape}')
print(f'坐标系  : {stack.crs}')
print(stack)

## 6. 计算并预览（加载到内存）

In [ ]:
print('正在从 Planetary Computer 下载数据...')
data = stack.compute()
print(f'下载完成，形状: {data.shape}')

# 提取各波段（第0个时间步，各波段）
B02 = data.sel(band='B02').values[0]
B03 = data.sel(band='B03').values[0]
B04 = data.sel(band='B04').values[0]
B08 = data.sel(band='B08').values[0]

print_array_info(B04, 'B04 Red')
print_array_info(B08, 'B08 NIR')

## 7. 计算遥感指数

In [ ]:
# Sentinel-2 反射率数值需除以 10000 归一化
bands_norm = {
    'blue' : B02 / 10000.0,
    'green': B03 / 10000.0,
    'red'  : B04 / 10000.0,
    'nir'  : B08 / 10000.0,
    'swir' : data.sel(band='B11').values[0] / 10000.0
}

indices = calculate_all_indices(bands_norm)

for name, arr in indices.items():
    valid = arr[np.isfinite(arr)]
    print(f'{name}: min={valid.min():.3f}  max={valid.max():.3f}  mean={valid.mean():.3f}')

## 8. 可视化

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle(f'NSW 测试区 Sentinel-2 指数\n{best.datetime.strftime("%Y-%m-%d")}', 
             fontsize=14)

plots = [
    ('NDVI', indices['NDVI'], 'RdYlGn', -0.2, 0.8),
    ('NDWI', indices['NDWI'], 'Blues',  -0.4, 0.4),
    ('BSI',  indices['BSI'],  'YlOrBr', -0.3, 0.3),
    ('SAVI', indices['SAVI'], 'RdYlGn', -0.2, 0.8),
]

for ax, (name, arr, cmap, vmin, vmax) in zip(axes.flat, plots):
    im = ax.imshow(arr, cmap=cmap, vmin=vmin, vmax=vmax)
    ax.set_title(name, fontsize=12)
    ax.axis('off')
    plt.colorbar(im, ax=ax, fraction=0.046)

plt.tight_layout()

fig_dir = ensure_dir('../results/figures')
fig_path = fig_dir / 'NSW_indices_2022_01.png'
plt.savefig(fig_path, dpi=150, bbox_inches='tight')
print(f'图像已保存: {fig_path}')
plt.show()

## 9. 保存为 GeoTIFF

In [ ]:
raw_dir = ensure_dir('../data/raw/sentinel2')

# 获取空间参考信息
crs       = stack.crs
transform = stack.transform
height, width = B04.shape

# 保存各指数为 GeoTIFF
for name, arr in indices.items():
    out_path = raw_dir / f'NSW_test_{name}_2022_01.tif'
    with rasterio.open(
        out_path, 'w',
        driver='GTiff',
        height=height, width=width,
        count=1,
        dtype='float32',
        crs=crs,
        transform=transform,
        compress='lzw'
    ) as dst:
        dst.write(arr.astype('float32'), 1)
    print(f'已保存: {out_path}')

print('\n✓ 笔记本 01 完成')
print('下一步: 运行 02_feature_engineering.ipynb')